In [3]:
# Verify GPU
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

# Uninstall and reinstall libraries to fix bitsandbytes and transformers issues
!pip uninstall -y transformers datasets peft torch pandas bitsandbytes accelerate
!pip install transformers==4.45.2 datasets==3.0.1 peft==0.13.2 torch==2.4.1 pandas==2.2.3 bitsandbytes==0.44.1 accelerate==1.0.1 --no-cache-dir

# Verify transformers version
import transformers
print("Transformers version:", transformers.__version__)

CUDA available: True
GPU name: Tesla T4
Found existing installation: transformers 4.45.2
Uninstalling transformers-4.45.2:
  Successfully uninstalled transformers-4.45.2
Found existing installation: datasets 3.0.1
Uninstalling datasets-3.0.1:
  Successfully uninstalled datasets-3.0.1
Found existing installation: peft 0.13.2
Uninstalling peft-0.13.2:
  Successfully uninstalled peft-0.13.2
Found existing installation: torch 2.4.1
Uninstalling torch-2.4.1:
  Successfully uninstalled torch-2.4.1
Found existing installation: pandas 2.2.3
Uninstalling pandas-2.2.3:
  Successfully uninstalled pandas-2.2.3
Found existing installation: bitsandbytes 0.44.1
Uninstalling bitsandbytes-0.44.1:
  Successfully uninstalled bitsandbytes-0.44.1
Found existing installation: accelerate 1.0.1
Uninstalling accelerate-1.0.1:
  Successfully uninstalled accelerate-1.0.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 12.7

Transformers version: 4.45.2


In [4]:
import pandas as pd

passages_df = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-bioasq/data/passages.parquet/part.0.parquet")
test_df = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-bioasq/data/test.parquet/part.0.parquet")
print("Test DataFrame size:", len(test_df))
print("Test columns:", test_df.columns)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Test DataFrame size: 4719
Test columns: Index(['question', 'answer', 'relevant_passage_ids'], dtype='object')


In [5]:
from datasets import Dataset

def format_data(row):
    return {"text": f"Question: {row['question']}\nAnswer: {row['answer']}"}

data = test_df.apply(format_data, axis=1).tolist()
dataset = Dataset.from_list(data)
dataset = dataset.train_test_split(test_size=0.2)
train_dataset = dataset["train"]
val_dataset = dataset["test"]
print("Train dataset size:", len(train_dataset))

Train dataset size: 3775


In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
import torch

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto",
    torch_dtype=torch.float16
)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

# Enable gradients for LoRA parameters
model.train()  # Set model to training mode
for name, param in model.named_parameters():
    if "lora" in name.lower():
        param.requires_grad = True

# Verify trainable parameters
model.print_trainable_parameters()
print("Trainable parameters with requires_grad=True:")
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044
Trainable parameters with requires_grad=True:
base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight
base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight
base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight
base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight
base_model.model.model.layers.1.self_attn.q_proj.lora_A.default.weight
base_model.model.model.layers.1.self_attn.q_proj.lora_B.default.weight
base_model.model.model.layers.1.self_attn.v_proj.lora_A.default.weight
base_model.model.model.layers.1.self_attn.v_proj.lora_B.default.weight
base_model.model.model.layers.2.self_attn.q_proj.lora_A.default.weight
base_model.model.model.layers.2.self_attn.q_proj.lora_B.default.weight
base_model.model.model.layers.2.self_attn.v_proj.lora_A.default.weight
base_model.model.model.layers.2.self_attn.v_proj.lora_B.default.weight
base_model.model.model.

In [7]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )

# Load tokenized datasets if saved, or tokenize anew
try:
    from datasets import load_from_disk
    tokenized_train = load_from_disk("./tokenized_train")
    tokenized_val = load_from_disk("./tokenized_val")
except:
    tokenized_train = train_dataset.map(tokenize_function, batched=True)
    tokenized_val = val_dataset.map(tokenize_function, batched=True)
    tokenized_train = tokenized_train.remove_columns(["text"])
    tokenized_val = tokenized_val.remove_columns(["text"])
    tokenized_train.set_format("torch")
    tokenized_val.set_format("torch")
    tokenized_train.save_to_disk("./tokenized_train")
    tokenized_val.save_to_disk("./tokenized_val")
print("Tokenized train sample:", tokenized_train[0])

Map:   0%|          | 0/3775 [00:00<?, ? examples/s]

Map:   0%|          | 0/944 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3775 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/944 [00:00<?, ? examples/s]

Tokenized train sample: {'input_ids': tensor([    1,   894, 29901,  1724,   338,   278,  1067,   493, 17056,   534,
         3873, 18556,  3829, 29973,    13, 22550, 29901,   450,  1067,   493,
        17056,   534,  3873,   295,   291, 29892,   607,   338,   263,  2211,
        29899,  1397,  3192, 12534, 29893, 10552, 29899,   845, 10501, 25745,
        13242,   962,   261, 29892,   338,   263,  4655,  4163,   297,   278,
        26823,  1302,  1446,   310,  3058,  1400, 29899, 29954,   324,  3146,
          322,  1095,   542,  3637,   293, 28999,  4027, 29889,     2,     2,
            2,     2,     2,     2,     2,     2,     2,     2,     2,     2,
            2,     2,     2,     2,     2,     2,     2,     2,     2,     2,
            2,     2,     2,     2,     2,     2,     2,     2,     2,     2,
            2,     2,     2,     2,     2,     2,     2,     2,     2,     2,
            2,     2,     2,     2,     2,     2,     2,     2,     2,     2,
            2,     2,     

In [8]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="./results",
    run_name="bioasq_tinyllama_finetune",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-4,
    fp16=True,
    logging_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    max_grad_norm=0.5,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
)

trainer.train()
trainer.save_model("./fine_tuned_model")
tokenizer.save_pretrained("./fine_tuned_model")

Epoch,Training Loss,Validation Loss
1,1.769600,1.741978
2,1.752500,1.736069
3,1.618100,1.741099


('./fine_tuned_model/tokenizer_config.json',
 './fine_tuned_model/special_tokens_map.json',
 './fine_tuned_model/tokenizer.model',
 './fine_tuned_model/added_tokens.json',
 './fine_tuned_model/tokenizer.json')

In [9]:
model.eval()
test_input = "Question: What is the main cause of lung cancer?\nAnswer:"
inputs = tokenizer(test_input, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=100)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)


Question: What is the main cause of lung cancer?
Answer: Lung cancer is the leading cause of cancer deaths worldwide. The most common cause of lung cancer is smoking. Other causes include exposure to asbestos, radon, and occupational exposure to chemicals. Lung cancer is also associated with exposure to air pollution.
The most common risk factors for lung cancer include smoking, exposure to asbestos, radon, and occupational exposure to chemicals.
Smoking is the most common


In [11]:
def chatbot():
    print("BioASQ Chatbot: Type 'quit' to exit.")
    while True:
        user_input = input("You: ")
        if user_input.lower() == "quit":
            break
        prompt = f"Question: {user_input}\nAnswer:"
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        outputs = model.generate(**inputs, max_new_tokens=100, do_sample=True)
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        print("Bot:", response.split("Answer:")[1].strip())

chatbot()

BioASQ Chatbot: Type 'quit' to exit.
You: quit


## Next Steps
### 1. Add RAG (Retrieval-Augmented Generation) using FAISS
### 2. Try QLoRA for lower memory footprint
### 3. Wrap chatbot with Gradio or Streamlit
### 4. Test with LLaMA2-7B if more powerful GPU is available

evaluation

In [12]:
!pip install rouge-score bert-score --quiet


In [13]:
from tqdm import tqdm

model.eval()

# Use a small set for faster evaluation (100 samples)
questions = test_df["question"].tolist()[:100]
true_answers = test_df["answer"].tolist()[:100]

generated_answers = []

for question in tqdm(questions):
    prompt = f"Question: {question}\nAnswer:"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=100)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = response.split("Answer:")[-1].strip()
    generated_answers.append(answer)


100%|██████████| 100/100 [10:55<00:00,  6.55s/it]


In [16]:
# Fix any non-string values
generated_answers = [str(ans) for ans in generated_answers]
true_answers = [str(ans) for ans in true_answers]


In [19]:
!pip install evaluate --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.3 MB/s eta 0:00:00


In [22]:
!pip install --force-reinstall torchvision --quiet


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 64.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 821.2/821.2 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 913.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 94.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.0/571.0 MB 867.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.2/158.2 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.

In [27]:
!pip install rouge-score --quiet


In [28]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

# Compute average ROUGE-L
scores = []
for ref, pred in zip(true_answers, generated_answers):
    score = scorer.score(ref, pred)
    scores.append(score["rougeL"].fmeasure)

# Final average ROUGE-L
avg_rouge_l = sum(scores) / len(scores)
print(" ROUGE-L Score (Manual):", round(avg_rouge_l, 4))


 ROUGE-L Score (Manual): 0.205


In [26]:
from bert_score import score

P, R, F1 = score(generated_answers, true_answers, lang="en", verbose=True)
print("🔹 BERT-F1 Score:", F1.mean().item())


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/4 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/2 [00:00<?, ?it/s]

done in 3.57 seconds, 28.04 sentences/sec
🔹 BERT-F1 Score: 0.8497596979141235


In [31]:
for i in range(3):
    print(f"\n Q: {questions[i]}")
    print(f" True: {true_answers[i]}")
    print(f" Pred: {generated_answers[i]}")



 Q: Is Hirschsprung disease a mendelian or a multifactorial disorder?
 True: Coding sequence mutations in RET, GDNF, EDNRB, EDN3, and SOX10 are involved in the development of Hirschsprung disease. The majority of these genes was shown to be related to Mendelian syndromic forms of Hirschsprung's disease, whereas the non-Mendelian inheritance of sporadic non-syndromic Hirschsprung disease proved to be complex; involvement of multiple loci was demonstrated in a multiplicative model.
 Pred: Hirschsprung disease is a mendelian disorder. It is a congenital defect of the intestinal tract that results from the absence of the third segment of the colon. It is a multifactorial disorder. It is caused by mutations in the gene encoding the intestinal epithelial cell adhesion molecule (E-cadherin). Hirschsprung disease is a multifactorial disorder. It is caused by mutations

 Q: List signaling molecules (ligands) that interact with the receptor EGFR?
 True: The 7 known EGFR ligands  are: epidermal 